# Teste de mesclagem de algoritmos - media ponderada de Ridge, Random Forest e Gradient Boosting

Este notebook e um **teste da Sprint 4**. Ele nao substitui nada do que foi entregue: apenas mede se
**combinar as previsoes de tres algoritmos** produz erro menor do que usar cada um deles sozinho.

## A pergunta

Cada um dos tres algoritmos erra de um jeito diferente. O Ridge e um modelo linear, e por isso acerta
a tendencia geral mas nao captura interacoes. Random Forest e Gradient Boosting sao conjuntos de
arvores, e capturam interacoes, mas de formas distintas: o primeiro faz a media de arvores
independentes, o segundo constroi arvores em sequencia, cada uma corrigindo o que sobrou da anterior.

Quando modelos erram de formas diferentes, a media das previsoes pode errar menos que qualquer um
deles isoladamente, porque parte dos erros se cancela. E isso que este notebook mede, **area por area**.

## O que e testado

Uma **grade de pesos** - e nao um peso fixo. Para cada area, a previsao combinada e

    previsao = peso_ridge x Ridge + peso_rf x RandomForest + peso_gb x GradientBoosting

com os tres pesos somando 1. Varias combinacoes sao testadas e o notebook reporta **qual delas deu o
menor erro em cada area**, comparando com o melhor modelo puro.

Ha uma hipotese de trabalho a verificar: a de que **dar peso maior ao Gradient Boosting melhora o
resultado**. Ela nao e assumida aqui. A grade inclui combinacoes com peso alto, medio e nulo para o
Gradient Boosting, e a conclusao sai da medicao, nao da expectativa.

## O que este teste **nao** faz

- Nao toca no **conjunto de teste reservado**. Todas as metricas vem da validacao cruzada sobre os
  ensaios de treino, exatamente como na entrega da Sprint 3.
- Nao gera nenhum arquivo de dados novo.
- Nao altera nenhum notebook existente.

## 1. Configuracao

Este notebook roda a bancada inteira **duas vezes**, sobre as duas versoes de tratamento das perdas:

- `dataset_<area>_13cat_snull.csv` - as perdas em **13 categorias**, uma coluna por combinacao
  declarada no motivo do ensaio.
- `dataset_<area>_perfil_snull.csv` - as mesmas perdas **agrupadas por perfil**, o que reduz as 13
  colunas a tres (`losses_miro`, `losses_simone`, `losses_no losses`).

O sufixo `_snull` e obrigatorio nas duas: sao as versoes com os nulos das colunas-alvo ja tratados na
preparacao. Usar uma base com nulos e a outra sem tornaria a comparacao entre as duas sem sentido,
porque cada uma estaria medindo sobre um conjunto de observacoes diferente.

A terceira peca e `particao_treino_teste.csv`, a particao fixa de treino e teste, com a coluna
`dobra` que define os grupos da validacao cruzada.

A particao nao e sorteada aqui: ela e lida de arquivo, para que todos os modelos sejam treinados e
avaliados exatamente sobre os mesmos ensaios - e, agora, para que as duas bases sejam comparadas sob
a mesma divisao. Sem essa fixacao, a variacao do sorteio contaminaria tanto a comparacao entre os
pesos quanto a comparacao entre as bases, que e justamente o que se quer medir.

A funcao `resolve()` procura cada arquivo nas pastas provaveis, de modo que o notebook roda tanto no
repositorio local quanto no Google Colab (neste caso, suba os arquivos pela barra lateral). A
particao nao esta na pasta `testes_S4`, e por isso a busca tambem varre `colabs_S3`.

A base de **referencia** e a `_13cat_snull`: e sobre ela que correm as secoes 3 a 9, que tratam da
mescla. A comparacao entre as duas bases esta na Secao 10, no fim.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os, glob, warnings

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# As duas versoes de tratamento das perdas que este notebook compara. Ambas na
# variante _snull, com os nulos das colunas-alvo ja tratados na preparacao.
SUFIXOS = ["_13cat_snull", "_perfil_snull"]

# Nome curto de cada base, para as tabelas.
NOME_BASE = {"_13cat_snull": "13 categorias", "_perfil_snull": "perfil de perdas"}

# Base de referencia: a que alimenta as secoes sobre a mescla. A outra entra na
# comparacao final.
SUFIXO = SUFIXOS[0]

AREAS = ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling",
         "main_vent_sys", "mainback", "oven_hmi"]

# Pastas candidatas, da mais especifica para a mais geral.
BASES = [
    ".", "..", os.path.join("..", ".."), "/content",
    "notebooks",
    os.path.join("notebooks", "colabs_S4", "testes_extras", "testes_S4"),
    os.path.join("..", "testes_S4"),
    os.path.join("notebooks", "colabs_S3"),
    os.path.join("..", "..", "..", "colabs_S3"),
    os.path.join("notebooks", "colabs_S3", "preparacao_testes"),
    os.path.join("..", "..", "..", "colabs_S3", "preparacao_testes"),
]


def resolve(arquivo):
    """Primeiro caminho existente para o arquivo, entre as bases candidatas."""
    tentativas = []
    for base in BASES:
        caminho = os.path.join(base, arquivo)
        tentativas.append(caminho)
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(
        "Arquivo nao encontrado: " + arquivo
        + ". Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(tentativas))


print("Diretorio atual:", os.path.abspath("."))
print("Versoes de tratamento das perdas:", ", ".join(SUFIXOS))
print("Base de referencia das secoes 3 a 9:", SUFIXO)
print("Areas a testar:", len(AREAS))

## 2. Leitura dos datasets e da particao

Cada dataset traz uma linha por ensaio. As colunas se dividem em dois grupos:

- **Entradas** - as configuracoes do ensaio, ja em *one-hot encoding*. Sao reconhecidas pelos
  prefixos `blower_direction_`, `housing_`, `board_position_`, `run_case_`, `losses_`,
  `deflector_hole_`, `material_coil_support_` e `cfg_`.
- **Alvos** - uma coluna por componente, com a temperatura medida em graus Celsius. Os valores
  ausentes estao aqui: sao medicoes que nao foram feitas naquele ensaio.

O prefixo `cfg_` merece registro: sao os atributos fisicos extraidos do motivo do ensaio durante a
preparacao. Eles constam da lista de entradas do `modelagem.ipynb`, e sem essa declaracao seriam
confundidos com colunas-alvo. A lista abaixo e a mesma daquele notebook. Ela serve as duas bases sem
alteracao: a diferenca entre `_13cat_snull` e `_perfil_snull` esta apenas em quantas colunas
`losses_` existem - treze na primeira, tres na segunda -, e nao no prefixo que as identifica.

A particao e lida do CSV, que tem as colunas `run_idx`, `conjunto` e `dobra`. Ensaios do conjunto de
teste tem `dobra` vazia - e assim que eles ficam de fora de tudo o que se faz aqui.

A leitura e feita para as duas bases de uma vez. A tabela de conferencia sai uma por base, e
a coluna `Nulos` deve ser zero nas duas: e o que o sufixo `_snull` garante.

In [ ]:
PREFIXOS_FEATURES = [
    "blower_direction", "housing", "board_position", "run_case",
    "losses", "deflector_hole", "material_coil_support",
    # Atributos fisicos extraidos do motivo do ensaio, na preparacao.
    # Sem este prefixo declarado, seriam tratados como colunas-alvo.
    "cfg",
]
COLUNAS_AUXILIARES = ["run_idx"]

# Um dicionario de datasets por base. A de referencia fica tambem em `datasets`,
# nome que as secoes seguintes usam.
datasets_por_base = {
    sufixo: {area: pd.read_csv(resolve(f"dataset_{area}{sufixo}.csv")) for area in AREAS}
    for sufixo in SUFIXOS
}
datasets = datasets_por_base[SUFIXO]

particao = pd.read_csv(resolve("particao_treino_teste.csv"))

# Dobra de cada ensaio. Ensaios do conjunto de teste reservado ficam com None e,
# por isso, sao excluidos de toda a validacao mais adiante.
DOBRA = {int(linha.run_idx): (None if pd.isna(linha.dobra) else int(linha.dobra))
         for linha in particao.itertuples()}
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})

n_treino = int((particao["conjunto"] == "treino").sum())
n_teste = int((particao["conjunto"] == "teste").sum())

print(f"Bases carregadas: {len(datasets_por_base)}  ({', '.join(SUFIXOS)})")
print(f"Ensaios de treino: {n_treino}  |  Ensaios de teste reservado: {n_teste}")
print(f"Dobras da validacao cruzada: {len(DOBRAS)}  {DOBRAS}")

for sufixo, base in datasets_por_base.items():
    print()
    print(f"--- {sufixo}  ({NOME_BASE[sufixo]}) ---")
    print("{:<16} {:>8} {:>9} {:>6} {:>7}".format("Area", "Ensaios", "Entradas", "Alvos", "Nulos"))
    print("-" * 50)
    for area, df in base.items():
        entradas = [c for c in df.columns if any(c.startswith(p + "_") for p in PREFIXOS_FEATURES)]
        alvos = [c for c in df.columns if c not in entradas + COLUNAS_AUXILIARES]
        print("{:<16} {:>8} {:>9} {:>6} {:>7}".format(
            area, len(df), len(entradas), len(alvos), int(df[alvos].isna().sum().sum())))

print()
print("A diferenca de 'Entradas' entre as duas bases sao as dez colunas losses_ a menos")
print("na versao por perfil: treze combinacoes agrupadas em tres perfis.")


## 3. A formulacao do modelo, reproduzida da Sprint 3

Para que o teste seja comparavel com o que ja foi entregue, a formulacao aqui e **a mesma da Secao 19
do `modelagem.ipynb`**, copiada sem alteracao de logica. Em resumo:

**Um modelo por area, e nao por componente.** Cada area vira uma unica tabela em *formato longo*, em
que cada linha e um par de ensaio e componente. Em vez de 74 modelos, um por componente, sao sete.

**Dois atributos descrevem o componente.** Ao vetor de configuracao somam-se o **patamar termico**
(a temperatura media daquele componente) e a **escala** (o desvio-padrao dele). Sao esses dois
numeros que dizem ao modelo de qual componente a linha trata, sem que ele precise memorizar 74
identidades por colunas indicadoras. A Sprint 3 mediu o que eles valem: sem eles o erro chega a
14,02 graus, so com o patamar cai para 7,17, e com os dois chega a 6,57.

**Esses dois atributos sao calculados somente sobre o treino de cada dobra.** E o ponto mais
delicado: patamar e escala sao estatisticas do proprio alvo, e calcula-los sobre a base inteira
levaria informacao da validacao para dentro do treino - vazamento. As funcoes abaixo recebem a
mascara de treino da dobra e so olham para ela.

Sobre os valores ausentes: o formato longo inclui apenas os pares ensaio-componente **efetivamente
medidos**. Linhas sem medicao nao entram nem no treino nem na validacao, de forma que nenhuma
imputacao e necessaria e nenhuma metrica e calculada sobre valor inventado.

In [ ]:
def separar(df, prefixos=PREFIXOS_FEATURES):
    """Devolve (X, cols_alvo).

    Colunas em COLUNAS_AUXILIARES (run_idx) sao excluidas de features e alvos:
    servem apenas para indexar a particao.
    """
    cols_entrada = [c for c in df.columns if any(c.startswith(p + "_") for p in prefixos)]
    cols_alvo = [c for c in df.columns if c not in cols_entrada + COLUNAS_AUXILIARES]
    return df[cols_entrada].copy(), cols_alvo


MIN_TREINO = 8
MIN_PREDICOES = 5


def formato_longo(df):
    """Uma linha por par de ensaio e componente medido.

    Copiada da Secao 19 do modelagem.ipynb. Quando a coluna run_idx nao existe
    no dataset, o indice posicional faz o seu papel -- os arquivos preservam a
    ordem dos ensaios da particao.
    """
    X, cols_alvo = separar(df)
    blocos = []
    for col in cols_alvo:
        y = df[col]
        if y.notna().sum() < MIN_PREDICOES:
            continue
        sub = X[y.notna()].copy()
        sub["_run"] = (df["run_idx"][y.notna()].to_numpy()
                       if "run_idx" in df.columns else np.arange(len(df))[y.notna()])
        sub["_componente"] = col
        sub["_alvo"] = y[y.notna()].to_numpy()
        blocos.append(sub)
    longo = pd.concat(blocos, ignore_index=True)
    longo["_dobra"] = longo["_run"].map(DOBRA)
    # Linhas sem dobra sao as do conjunto de teste reservado, que fica de fora.
    return longo[longo["_dobra"].notna()].reset_index(drop=True)


def atributos_do_componente(longo, treino):
    """Patamar e escala de cada componente, medidos so no treino da dobra."""
    g = longo[treino].groupby("_componente")["_alvo"]
    media = g.mean()
    desvio = g.std().fillna(1.0).replace(0, 1.0)
    geral = float(longo.loc[treino, "_alvo"].mean())
    return media, desvio, geral


def matriz(longo, mascara, entradas, media, desvio, geral):
    """Vetor de configuracao acrescido do patamar e da escala do componente."""
    comp = longo.loc[mascara, "_componente"]
    return np.hstack([
        longo.loc[mascara, entradas].to_numpy(),
        comp.map(media).fillna(geral).to_numpy().reshape(-1, 1),
        comp.map(desvio).fillna(1.0).to_numpy().reshape(-1, 1),
    ])


print("{:<16} {:>10} {:>12} {:>12}".format("Area", "Linhas", "Componentes", "Colunas"))
print("-" * 54)
for area, df in datasets.items():
    longo = formato_longo(df)
    entradas = [c for c in longo.columns if not c.startswith("_")]
    print("{:<16} {:>10} {:>12} {:>12}".format(
        area, len(longo), longo["_componente"].nunique(), len(entradas) + 2))
print()
print("A coluna 'Colunas' ja inclui os dois atributos do componente.")

## 4. Metricas

As tres metricas reportadas sao as de sempre:

- **MAE** - erro absoluto medio, em graus Celsius. E o numero principal: quanto o modelo erra, em
  media, numa leitura qualquer.
- **RMSE** - raiz do erro quadratico medio, tambem em graus. Pune erros grandes mais do que o MAE, e
  por isso a distancia entre os dois indica se os erros estao concentrados em poucos casos.
- **R2** - fracao da variacao do alvo que o modelo explica. Negativo significa que o modelo e pior
  do que simplesmente chutar a media historica daquele componente.

As tres sao calculadas **por componente** e depois promediadas dentro da area (macro-media). Essa
escolha da o mesmo peso a cada componente, e responde se o modelo funciona bem em todas as partes da
area - e nao apenas nas mais medidas.

O R2 sai ausente quando o alvo de um componente assume um unico valor, caso em que ele e indefinido
e a implementacao do scikit-learn devolveria zero, valor que se confunde com um resultado legitimo.
A raiz do erro quadratico e calculada de forma explicita, porque o parametro que a produzia foi
removido do scikit-learn na versao 1.9.

In [ ]:
def metricas_por_componente(real, previsto):
    """MAE, RMSE e R2 de um componente, com o alvo em graus Celsius."""
    real = np.asarray(real, dtype=float).ravel()
    previsto = np.asarray(previsto, dtype=float).ravel()
    if real.shape != previsto.shape:
        raise ValueError(f"dimensoes diferentes: {real.shape} e {previsto.shape}")
    if real.size == 0:
        raise ValueError("nenhuma observacao a avaliar")
    if np.isnan(real).any() or np.isnan(previsto).any():
        raise ValueError("ha valores ausentes; a mascara deve ser aplicada antes")

    erro = previsto - real
    mae = float(np.mean(np.abs(erro)))
    # A raiz e explicita: o parametro squared do mean_squared_error foi removido
    # do scikit-learn na versao 1.9.
    rmse = float(np.sqrt(np.mean(erro ** 2)))

    if np.unique(real).size < 2:
        r2 = float("nan")   # indefinido quando o alvo assume um unico valor
    else:
        r2 = 1.0 - float(np.sum(erro ** 2)) / float(np.sum((real - real.mean()) ** 2))

    return mae, rmse, r2


def macro_da_area(g, previsto):
    """Media das metricas sobre os componentes da area."""
    previsto = pd.Series(np.asarray(previsto, dtype=float), index=g.index)
    maes, rmses, r2s = [], [], []
    for _, bloco in g.groupby("componente"):
        mae, rmse, r2 = metricas_por_componente(bloco["real"], previsto.loc[bloco.index])
        maes.append(mae)
        rmses.append(rmse)
        if not np.isnan(r2):
            r2s.append(r2)
    return {
        "MAE": float(np.mean(maes)),
        "RMSE": float(np.mean(rmses)),
        "R2": float(np.mean(r2s)) if r2s else float("nan"),
        "componentes": len(maes),
        "n": int(len(g)),
    }


print("Metricas definidas: metricas_por_componente, macro_da_area")

## 5. Treino dos tres algoritmos

A validacao e **leave-one-group-out** sobre a coluna `dobra`: a cada rodada, uma configuracao inteira
sai do treino e serve de validacao. O conjunto de teste reservado nao participa de nada.

Dentro de cada dobra, os tres algoritmos sao treinados sobre exatamente a mesma matriz e as previsoes
dos tres sao guardadas lado a lado. E essa tabela de previsoes que permite testar quantas combinacoes
de pesos se queira sem retreinar nada.

### Por que so o Ridge recebe escalonamento

O Ridge e um modelo linear com penalizacao sobre os coeficientes. A penalizacao e a mesma para todos
eles, de modo que uma entrada em escala maior que as outras - o patamar termico, que chega a centenas
de graus, ao lado de colunas binarias de zero e um - acaba penalizada de forma diferente. Sem
padronizar, o Ridge fica prejudicado e a mescla seria injusta com ele.

Arvores nao tem esse problema: elas decidem por cortes em cada coluna separadamente, e multiplicar
uma coluna por uma constante nao muda corte algum. Random Forest e Gradient Boosting, por isso,
recebem as entradas **sem escalonar**.

O `StandardScaler` e ajustado **apenas sobre o treino da dobra** e depois aplicado a validacao.
Ajusta-lo sobre a base inteira levaria a media e o desvio dos ensaios de validacao para dentro do
treino, que e exatamente o vazamento que a validacao cruzada existe para evitar.

### Hiperparametros

Os do `modelagem.ipynb`, onde ele os especifica:

- **Ridge** - `RidgeCV` com a mesma grade de `alpha` da Secao 7, escolhido por validacao interna
  dentro de cada dobra.
- **Random Forest** - 300 arvores, `random_state=42`, demais parametros no padrao. Registre-se que
  `max_features` nao e alterado: o padrao de regressao considera todas as colunas em cada divisao, e
  a alternativa custou 1,55 grau na medicao da Sprint 3.
- **Gradient Boosting** - 300 arvores e taxa de aprendizado de 0,05, que sao os valores registrados
  na Secao 19 do `modelagem.ipynb` para esta mesma formulacao.

Esta celula e a mais demorada do notebook - sete areas vezes dezessete dobras vezes tres algoritmos,
alguns minutos ao todo.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

ALPHAS_RIDGE = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
RF_PARAMS = dict(n_estimators=300, random_state=42, n_jobs=-1)
GB_PARAMS = dict(n_estimators=300, learning_rate=0.05, random_state=42)

MODELOS = ["ridge", "rf", "gb"]
NOME_LONGO = {"ridge": "Ridge", "rf": "Random Forest", "gb": "Gradient Boosting"}


def treinar(base):
    """Previsoes de validacao dos tres algoritmos, para um dicionario de datasets.

    A rotina e a mesma de antes, so que parametrizada pela base: assim as duas
    versoes de tratamento das perdas passam exatamente pelo mesmo caminho, com as
    mesmas dobras e as mesmas sementes. O que diferir entre elas no resultado vem
    da codificacao das perdas, e nao do procedimento.
    """
    blocos_previsao = []
    for area, df in base.items():
        longo = formato_longo(df)
        entradas = [c for c in longo.columns if not c.startswith("_")]

        for d in DOBRAS:
            treino = (longo["_dobra"] != d).to_numpy()
            validacao = (longo["_dobra"] == d).to_numpy()
            if treino.sum() < MIN_TREINO or not validacao.any():
                continue

            # Patamar e escala: so o treino desta dobra e consultado.
            media, desvio, geral = atributos_do_componente(longo, treino)
            X_tr = matriz(longo, treino, entradas, media, desvio, geral)
            X_val = matriz(longo, validacao, entradas, media, desvio, geral)
            y_tr = longo.loc[treino, "_alvo"]

            # Escalonamento apenas para o Ridge, ajustado so no treino da dobra.
            escala = StandardScaler().fit(X_tr)
            p_ridge = RidgeCV(alphas=ALPHAS_RIDGE).fit(
                escala.transform(X_tr), y_tr).predict(escala.transform(X_val))

            # Arvores usam as entradas sem escalonar.
            p_rf = RandomForestRegressor(**RF_PARAMS).fit(X_tr, y_tr).predict(X_val)
            p_gb = GradientBoostingRegressor(**GB_PARAMS).fit(X_tr, y_tr).predict(X_val)

            bloco = longo.loc[validacao, ["_componente", "_run", "_alvo"]].copy()
            bloco.columns = ["componente", "ensaio", "real"]
            bloco["dataset"] = area
            bloco["dobra"] = d
            bloco["ridge"], bloco["rf"], bloco["gb"] = p_ridge, p_rf, p_gb
            blocos_previsao.append(bloco)

    return pd.concat(blocos_previsao, ignore_index=True)


# As duas bases sao treinadas aqui. A de referencia fica tambem em `predicoes`.
predicoes_por_base = {sufixo: treinar(base) for sufixo, base in datasets_por_base.items()}
predicoes = predicoes_por_base[SUFIXO]

for sufixo, p in predicoes_por_base.items():
    print(f"--- {sufixo}  ({NOME_BASE[sufixo]}) ---")
    print(f"  Previsoes de validacao: {len(p)}  |  componentes: {p['componente'].nunique()}"
          f"  |  ensaios: {p['ensaio'].nunique()}")
    for m in MODELOS:
        print(f"  {NOME_LONGO[m]:<18} {np.abs(p[m] - p['real']).mean():.3f} C")
    print()


### Conferencia do isolamento do conjunto de teste

Antes de medir qualquer coisa, vale confirmar por numero o que a formulacao promete: nenhum ensaio
reservado apareceu na validacao.

In [ ]:
ensaios_teste = {int(l.run_idx) for l in particao.itertuples() if l.conjunto == "teste"}

print(f"Ensaios do teste reservado: {len(ensaios_teste)}")
falhou = False
for sufixo, p in predicoes_por_base.items():
    invasores = sorted(set(p["ensaio"].astype(int)) & ensaios_teste)
    falhou = falhou or bool(invasores)
    print(f"  {sufixo:<16} reservados que apareceram na validacao: {len(invasores)}"
          + ("" if not invasores else f"  -- {invasores}"))
print("Verificacao:", "FALHOU" if falhou else "PASSOU")


## 6. A grade de pesos

Cada combinacao e uma tripla `(peso_ridge, peso_rf, peso_gb)` que soma 1. A grade cobre tres regioes:

- **Sem o Ridge**, variando a divisao entre as duas familias de arvores: `(0, 0.5, 0.5)`,
  `(0, 0.4, 0.6)` e `(0, 0.3, 0.7)`. E aqui que a hipotese sobre o Gradient Boosting e testada
  diretamente - se ela valer, o erro deve cair conforme o peso dele sobe.
- **Com um pouco de Ridge**: `(0.2, 0.4, 0.4)`, `(0.1, 0.3, 0.6)` e a media simples
  `(0.33, 0.33, 0.34)`. Um modelo linear fraco pode ainda assim ajudar, se errar em direcao oposta a
  das arvores.
- **Os tres modelos puros**: `(1, 0, 0)`, `(0, 1, 0)` e `(0, 0, 1)`. Sao a referencia sem a qual nao
  se sabe se a mescla trouxe algum ganho.

Como as previsoes ja estao guardadas, testar uma combinacao custa uma multiplicacao - nada e
retreinado. Vale registrar a contrapartida: os pesos sao escolhidos olhando para o mesmo conjunto de
validacao sobre o qual sao avaliados, de modo que o ganho reportado e uma estimativa otimista. A
Secao 10 volta a esse ponto.

In [ ]:
GRADE_PESOS = [
    (0.00, 0.50, 0.50),
    (0.00, 0.40, 0.60),
    (0.00, 0.30, 0.70),
    (0.20, 0.40, 0.40),
    (0.10, 0.30, 0.60),
    (0.33, 0.33, 0.34),
    (1.00, 0.00, 0.00),   # Ridge puro
    (0.00, 1.00, 0.00),   # Random Forest puro
    (0.00, 0.00, 1.00),   # Gradient Boosting puro
]

NOMES_PUROS = {
    (1.00, 0.00, 0.00): "Ridge puro",
    (0.00, 1.00, 0.00): "Random Forest puro",
    (0.00, 0.00, 1.00): "Gradient Boosting puro",
}

for pesos in GRADE_PESOS:
    if abs(sum(pesos) - 1.0) > 1e-9:
        raise ValueError(f"os pesos {pesos} nao somam 1")


def rotulo(pesos):
    base = "ridge {:.2f} | rf {:.2f} | gb {:.2f}".format(*pesos)
    return base + ("   <- " + NOMES_PUROS[pesos] if pesos in NOMES_PUROS else "")


print(f"Combinacoes de pesos a testar: {len(GRADE_PESOS)}")
for pesos in GRADE_PESOS:
    print(" ", rotulo(pesos))

## 7. Resultado por area

A tabela de cada area lista todas as combinacoes testadas, ordenadas do menor para o maior MAE. As
linhas dos modelos puros vem marcadas, para que se veja de imediato se alguma mescla superou o melhor
deles - e por quanto.

In [ ]:
def tabela_resultados(predicoes):
    """Uma linha por par de area e combinacao de pesos, com as metricas macro."""
    linhas = []
    for area, g in predicoes.groupby("dataset"):
        P = g[MODELOS].to_numpy()
        for pesos in GRADE_PESOS:
            combinada = P @ np.array(pesos, dtype=float)
            linhas.append({
                "area": area,
                "pesos": rotulo(pesos),
                "w_ridge": pesos[0], "w_rf": pesos[1], "w_gb": pesos[2],
                "puro": NOMES_PUROS.get(pesos, ""),
                **macro_da_area(g, combinada),
            })
    return pd.DataFrame(linhas)


resultados_por_base = {sufixo: tabela_resultados(p)
                       for sufixo, p in predicoes_por_base.items()}
resultados = resultados_por_base[SUFIXO]

print(f"Linhas de resultado por base: {len(resultados)}  "
      f"({len(AREAS)} areas x {len(GRADE_PESOS)} combinacoes de pesos)")
print(f"Bases avaliadas: {', '.join(resultados_por_base)}")


In [ ]:
for area in AREAS:
    bloco = resultados[resultados["area"] == area].sort_values("MAE")
    primeira = bloco.iloc[0]
    print("=" * 76)
    print(f"{area}   |   {int(primeira['componentes'])} componentes   |   "
          f"{int(primeira['n'])} previsoes de validacao")
    print("=" * 76)
    print(bloco[["pesos", "MAE", "RMSE", "R2"]].round(3).to_string(index=False))
    print()

## 8. Tabela resumo - a mescla vale a pena?

A comparacao que interessa e entre **o melhor peso encontrado** e **o melhor modelo puro** da mesma
area. O ganho e a diferenca entre os dois MAEs, em graus: positivo significa que a mescla errou menos.

A coluna em porcentagem e o que permite comparar areas de dificuldade muito diferente - um decimo de
grau significa uma coisa no Oven HMI, cujo erro e de poucos graus, e outra bem diferente no Coil
Temps. Quando `mescla_venceu` e falso, o melhor resultado da area foi um modelo puro e o ganho e zero
por construcao.

In [ ]:
resumo = []
for area in AREAS:
    bloco = resultados[resultados["area"] == area]
    melhor = bloco.loc[bloco["MAE"].idxmin()]
    puros = bloco[bloco["puro"] != ""]
    melhor_puro = puros.loc[puros["MAE"].idxmin()]
    ganho = float(melhor_puro["MAE"] - melhor["MAE"])
    resumo.append({
        "area": area,
        "melhor_peso": melhor["pesos"].split("   <-")[0],
        "MAE_melhor": float(melhor["MAE"]),
        "melhor_puro": melhor_puro["puro"],
        "MAE_puro": float(melhor_puro["MAE"]),
        "ganho_C": ganho,
        "ganho_%": 100.0 * ganho / float(melhor_puro["MAE"]),
        "mescla_venceu": melhor["puro"] == "",
    })

df_resumo = pd.DataFrame(resumo)
print(df_resumo.round(3).to_string(index=False))
print()

venceu = int(df_resumo["mescla_venceu"].sum())
print(f"Areas em que a mescla superou todos os modelos puros: {venceu} de {len(AREAS)}")
print(f"Ganho medio sobre o melhor puro: {df_resumo['ganho_C'].mean():+.3f} C "
      f"({df_resumo['ganho_%'].mean():+.2f}%)")

### A hipotese do peso maior para o Gradient Boosting

A hipotese de trabalho era que dar peso maior ao Gradient Boosting melhoraria o resultado. A celula
abaixo a confronta com a medicao de duas formas.

A primeira olha apenas para as tres combinacoes sem Ridge, em que o peso do Gradient Boosting sobe de
0,5 para 0,7: se a hipotese valer, o MAE deve cair de forma consistente ao longo dessa serie.

A segunda compara os dois modelos de arvore sozinhos, area por area. Se o Gradient Boosting puro ja
erra mais que o Random Forest puro, nao ha razao para esperar que aumentar o peso dele melhore a
mescla - e e esse o teste direto da hipotese.

In [ ]:
serie = [(0.00, 0.50, 0.50), (0.00, 0.40, 0.60), (0.00, 0.30, 0.70)]

print("MAE por area ao longo da serie sem Ridge, com peso do GB crescente:")
print(("{:<16}" + " {:>10}" * len(serie) + " {:>12}").format(
    "area", *[f"gb {p[2]:.1f}" for p in serie], "tendencia"))
print("-" * 64)

descendo = 0
for area in AREAS:
    bloco = resultados[resultados["area"] == area]
    valores = [float(bloco[(bloco["w_ridge"] == p[0]) & (bloco["w_rf"] == p[1])
                           & (bloco["w_gb"] == p[2])]["MAE"].iloc[0]) for p in serie]
    caindo = all(b < a for a, b in zip(valores, valores[1:]))
    descendo += int(caindo)
    print(("{:<16}" + " {:>10.3f}" * len(serie) + " {:>12}").format(
        area, *valores, "cai" if caindo else "nao cai"))

print()
print(f"Areas em que o MAE cai ao aumentar o peso do GB: {descendo} de {len(AREAS)}")
print()

print("Modelos de arvore sozinhos, por area:")
print("{:<16} {:>10} {:>10} {:>12}".format("area", "RF puro", "GB puro", "melhor"))
print("-" * 52)
gb_melhor = 0
for area in AREAS:
    bloco = resultados[resultados["area"] == area]
    mae_rf = float(bloco[bloco["puro"] == "Random Forest puro"]["MAE"].iloc[0])
    mae_gb = float(bloco[bloco["puro"] == "Gradient Boosting puro"]["MAE"].iloc[0])
    gb_melhor += int(mae_gb < mae_rf)
    print("{:<16} {:>10.3f} {:>10.3f} {:>12}".format(
        area, mae_rf, mae_gb, "GB" if mae_gb < mae_rf else "RF"))

print()
print(f"Areas em que o Gradient Boosting puro supera o Random Forest puro: "
      f"{gb_melhor} de {len(AREAS)}")
print()
if descendo == 0 and gb_melhor == 0:
    print("CONCLUSAO: a hipotese NAO se confirma. Aumentar o peso do Gradient Boosting piora")
    print("o resultado em todas as areas, e o proprio GB sozinho erra mais que o RF sozinho.")
elif descendo == len(AREAS) and gb_melhor == len(AREAS):
    print("CONCLUSAO: a hipotese se confirma em todas as areas.")
else:
    print("CONCLUSAO: a hipotese se confirma apenas em parte das areas. Ela nao vale como")
    print("regra geral, e o peso deve ser decidido area por area, pela tabela acima.")

## 9. Verificacao de sanidade - o Random Forest puro bate com a Sprint 3?

Este teste so tem valor se a formulacao tiver sido reproduzida corretamente. A checagem e direta: o
**Random Forest puro** desta bancada e o modelo entregue na Sprint 3, e seu MAE por area deve ficar
proximo do que a entrega registrou (tabela da Secao 19 da documentacao).

Duas ressalvas sobre a comparacao, que explicam por que se espera aproximacao e nao igualdade:

1. Os valores de referencia foram medidos sobre a base **`_perfil`**, que e a adotada na entrega.
   Aqui a base e a **`_13cat`**. A propria Sprint 3 mediu a diferenca entre as duas sob o mesmo
   protocolo e encontrou 7,560 contra 7,543 graus no agregado - diferenca pequena, mas nao nula, e
   que pode ser maior dentro de uma area especifica.
2. Os arquivos `_13cat` da pasta `testes_S4` sao uma regeracao posterior a da Sprint 3, com
   tratamento de nulos revisado.

Por isso a tolerancia abaixo e uma **faixa**, e nao uma igualdade. O alerta dispara quando a
diferenca passa de 1,5 grau **e** de 30% do valor de referencia - as duas condicoes ao mesmo tempo,
para nao acusar por variacao trivial uma area cujo erro ja e pequeno.

In [ ]:
# MAE por area do Random Forest sob a formulacao por area, registrado na
# documentacao da Sprint 3 (base _perfil, mesmas dobras).
REFERENCIA_SPRINT3 = {
    "oven_hmi": 2.42,
    "ct_knobs": 3.16,
    "ipc_cooling": 4.04,
    "main_vent_sys": 4.68,
    "mainback": 5.68,
    "ipc_comp": 7.30,
    "coil_temps": 11.71,
}

TOL_ABS, TOL_REL = 1.5, 0.30

print("{:<16} {:>10} {:>12} {:>12} {:>10}  {}".format(
    "area", "RF aqui", "Sprint 3", "diferenca", "dif %", "situacao"))
print("-" * 76)

alertas = []
for area in AREAS:
    bloco = resultados[(resultados["area"] == area)
                       & (resultados["puro"] == "Random Forest puro")]
    aqui = float(bloco["MAE"].iloc[0])
    ref = REFERENCIA_SPRINT3[area]
    dif = aqui - ref
    dif_rel = dif / ref
    fora = abs(dif) > TOL_ABS and abs(dif_rel) > TOL_REL
    if fora:
        alertas.append(area)
    print("{:<16} {:>10.2f} {:>12.2f} {:>+12.2f} {:>+9.1f}%  {}".format(
        area, aqui, ref, dif, 100 * dif_rel, "DIVERGE" if fora else "compativel"))

print()
if alertas:
    print("ATENCAO: o MAE do Random Forest puro diverge da referencia da Sprint 3 nas areas:")
    print("  " + ", ".join(alertas))
    print("A divergencia pode vir da diferenca entre as bases _perfil e _13cat, ou da regeracao")
    print("dos arquivos em testes_S4 -- mas tambem pode indicar que a formulacao foi reproduzida")
    print("de forma errada. Enquanto isso nao for esclarecido, os resultados de mescla acima NAO")
    print("devem ser lidos como comparaveis com a entrega da Sprint 3.")
else:
    print("VERIFICACAO PASSOU: o Random Forest puro reproduz o desempenho da entrega da Sprint 3")
    print("em todas as areas, dentro da tolerancia. A formulacao esta reproduzida corretamente e")
    print("a comparacao entre as combinacoes de pesos e valida.")

## 10. As perdas agrupadas por perfil treinam melhor ou pior?

Ate aqui tudo correu sobre a base `_13cat_snull`. Esta secao repete a leitura sobre a
`_perfil_snull` e poe as duas lado a lado.

A diferenca entre elas e apenas a **codificacao das perdas**. Na versao de 13 categorias, cada
combinacao de perdas declarada no motivo do ensaio vira uma coluna propria - sao 13 colunas, varias
delas acesas em pouquissimos ensaios. Na versao por perfil, essas 13 combinacoes sao agrupadas nos
tres perfis que as originam (`miro`, `simone` e `no losses`), e restam tres colunas. Todo o resto -
ensaios, componentes, dobras, algoritmos, sementes - e identico.

O que se espera de um agrupamento assim e a troca de sempre: perde-se detalhe e ganha-se densidade.
Uma coluna acesa em tres ensaios quase nao tem de onde tirar um padrao; a mesma informacao diluida em
uma coluna acesa em vinte ensaios tem. Se o detalhe descartado nao carregava sinal, o agrupamento
ajuda. Se carregava, atrapalha. Qual dos dois acontece e o que as tabelas abaixo medem, e a resposta
nao precisa ser a mesma em todas as areas.

### O que a comparacao mede, e o que ela nao mede

Os numeros abaixo comparam o **erro da validacao cruzada** das duas bases. Duas ressalvas valem o
registro, porque a tabela sozinha nao as mostra:

1. **O sufixo `_snull` imputa valores, e isso desloca o nivel do erro.** As colunas-alvo com
   medicao ausente foram preenchidas na preparacao, e essas linhas imputadas entram tanto no treino
   quanto na validacao. Parte do erro medido aqui e, portanto, erro contra um valor que nao foi
   medido em bancada - e nao e pouco. No `coil_temps`, a area mais afetada, o Random Forest puro
   erra **11,93 graus** sobre a base `_perfil` com os nulos ainda ausentes (344 linhas de validacao)
   e **14,87** sobre a `_perfil_snull` (368 linhas): as 24 linhas imputadas que entram respondem
   sozinhas por quase tres graus de erro medio. O mesmo vale para a `_13cat`, que vai de 11,37 para
   14,57.

   A consequencia pratica e dupla. A **comparacao** entre as duas bases continua valida, porque as
   duas recebem exatamente a mesma imputacao nas mesmas linhas. Mas o **nivel** do erro relatado
   aqui nao e comparavel com o da entrega da Sprint 3, nem com o de qualquer teste feito sobre
   medicoes apenas - e nao deve ser citado como o erro do modelo.
2. **A melhor combinacao de pesos foi escolhida sobre a mesma validacao que a avalia**, nas duas
   bases. A tabela da melhor mescla herda esse otimismo. E por isso que a comparacao principal, logo
   abaixo, e feita sobre os **modelos puros**, que nao passam por escolha alguma.

### Os algoritmos puros, nas duas bases

In [ ]:
# MAE agregado de cada algoritmo puro, nas duas bases. Esta e a comparacao mais
# limpa que ha aqui: nenhum peso foi escolhido, nada foi otimizado.
print("MAE agregado sobre todas as previsoes de validacao, por base e algoritmo:")
print()
print("{:<20} {:>14} {:>16} {:>12}".format("algoritmo", "13 categorias", "perfil de perdas", "diferenca"))
print("-" * 66)

for m in MODELOS:
    a = float(np.abs(predicoes_por_base["_13cat_snull"][m]
                     - predicoes_por_base["_13cat_snull"]["real"]).mean())
    b = float(np.abs(predicoes_por_base["_perfil_snull"][m]
                     - predicoes_por_base["_perfil_snull"]["real"]).mean())
    print("{:<20} {:>14.3f} {:>16.3f} {:>+12.3f}".format(NOME_LONGO[m], a, b, b - a))

print()
print("A coluna 'diferenca' e perfil menos 13 categorias: negativo significa que o")
print("agrupamento por perfil errou MENOS, ou seja, treinou melhor.")


### Area por area

In [ ]:
# Por area, comparando duas coisas: o melhor modelo puro (sem escolha de pesos) e
# a melhor mescla (com escolha, e por isso otimista nas duas bases).
def melhores_da_area(res, area):
    bloco = res[res["area"] == area]
    puros = bloco[bloco["puro"] != ""]
    melhor_puro = puros.loc[puros["MAE"].idxmin()]
    melhor = bloco.loc[bloco["MAE"].idxmin()]
    return float(melhor_puro["MAE"]), float(melhor["MAE"])


LIMIAR = 0.10   # graus; o mesmo limiar de relevancia usado nas secoes anteriores

comparacao_bases = []
for area in AREAS:
    puro_13, mescla_13 = melhores_da_area(resultados_por_base["_13cat_snull"], area)
    puro_pf, mescla_pf = melhores_da_area(resultados_por_base["_perfil_snull"], area)
    dif = puro_pf - puro_13
    comparacao_bases.append({
        "area": area,
        "puro_13cat": puro_13,
        "puro_perfil": puro_pf,
        "dif_puro": dif,
        "dif_%": 100.0 * dif / puro_13,
        "mescla_13cat": mescla_13,
        "mescla_perfil": mescla_pf,
        "veredito": ("perfil MELHOR" if dif <= -LIMIAR
                     else ("perfil pior" if dif >= LIMIAR else "empate")),
    })

df_bases = pd.DataFrame(comparacao_bases)
print(df_bases.round(3).to_string(index=False))
print()

melhor = int((df_bases["veredito"] == "perfil MELHOR").sum())
pior = int((df_bases["veredito"] == "perfil pior").sum())
empate = int((df_bases["veredito"] == "empate").sum())
print(f"Areas em que o perfil treina melhor: {melhor}   pior: {pior}   empate: {empate}"
      f"   (limiar de {LIMIAR:.2f} C)")
print(f"Diferenca media sobre as sete areas: {df_bases['dif_puro'].mean():+.3f} C"
      f"  ({df_bases['dif_%'].mean():+.1f}%)")
print()
print("Lembrando: a media sobre as areas pesa igual uma area de erro grande e uma de")
print("erro pequeno. Por isso o veredito por area importa mais do que a media.")


### O veredito

In [ ]:
# O veredito, escrito por extenso a partir dos numeros acima -- nao a mao.
media_dif = float(df_bases["dif_puro"].mean())
piores = df_bases[df_bases["veredito"] == "perfil pior"]["area"].tolist()
melhores = df_bases[df_bases["veredito"] == "perfil MELHOR"]["area"].tolist()

print("=" * 76)
print("VEREDITO: o treinamento com as perdas agrupadas por perfil")
print("=" * 76)
print()

# Duas leituras do mesmo dado, que podem discordar -- e quando discordam, isso
# e a informacao principal, nao um detalhe a esconder.
if melhor > pior:
    contagem = f"vence em {melhor} das {len(AREAS)} areas e perde em {pior}"
    lado_contagem = "melhor"
elif pior > melhor:
    contagem = f"perde em {pior} das {len(AREAS)} areas e vence em {melhor}"
    lado_contagem = "pior"
else:
    contagem = f"vence em {melhor} areas e perde em {pior}"
    lado_contagem = "empate"

lado_media = ("melhor" if media_dif <= -0.01
              else ("pior" if media_dif >= 0.01 else "empate"))

print(f"Por contagem de areas:   {contagem}.")
print(f"Por diferenca media:     {media_dif:+.3f} C no melhor modelo puro"
      f"  ({df_bases['dif_%'].mean():+.1f}% em media relativa).")
print()

FRASE = {"melhor": "favorece o perfil",
         "pior": "favorece as 13 categorias",
         "empate": "nao separa as duas"}

if lado_contagem == lado_media:
    if lado_contagem == "empate":
        print("As duas leituras concordam: nao ha diferenca que se sustente entre as bases.")
    else:
        print(f"As duas leituras concordam: o agrupamento por perfil treina "
              f"{lado_contagem.upper()}.")
else:
    print("ATENCAO: as duas leituras NAO apontam para o mesmo lado, e por isso o")
    print("resultado nao deve ser resumido num veredito unico.")
    print(f"  - a contagem de areas {FRASE[lado_contagem]};")
    print(f"  - a diferenca media {FRASE[lado_media]}.")
    print("Quando isso acontece, o motivo esta no tamanho dos ganhos e das perdas, e")
    print("nao na quantidade delas: um punhado de areas que perde por muito desloca a")
    print("media inteira. Os dois numeros abaixo mostram exatamente isso.")

print()
if melhores:
    ganho = df_bases[df_bases["area"].isin(melhores)]["dif_puro"]
    print(f"Melhor com perfil ({len(melhores)}): " + ", ".join(melhores))
    print(f"   ganho medio onde ganha: {-ganho.mean():.3f} C")
if piores:
    perda = df_bases[df_bases["area"].isin(piores)]["dif_puro"]
    print(f"Pior com perfil ({len(piores)}):   " + ", ".join(piores))
    print(f"   perda media onde perde: {perda.mean():.3f} C")

print()
print("A leitura util nao e um sim ou nao global. O agrupamento por perfil troca")
print("detalhe por densidade: cada coluna de perdas passa a ser vista em mais")
print("ensaios, e em troca some a distincao entre combinacoes. Onde as 13")
print("categorias tinham colunas acesas em pouquissimos ensaios, essa troca")
print("compensa; onde a distincao entre combinacoes carregava sinal de verdade,")
print("ela apaga esse sinal. A decisao, portanto, e por area -- e a tabela acima")
print("diz quais areas ficam de cada lado.")


## 11. Como ler este resultado

Tres pontos delimitam o que este teste autoriza concluir.

**O peso foi escolhido sobre a mesma validacao que o avalia.** Nove combinacoes foram comparadas em
cada area e a melhor foi reportada. Escolher a melhor entre nove sobre um mesmo conjunto tende a
capturar parte do ruido junto com o sinal, e por isso o ganho medido e uma estimativa otimista do
ganho real. Um ganho de poucos centesimos de grau, em particular, nao deve ser tratado como diferenca
de desempenho.

**O conjunto de teste reservado continua intacto.** Nenhum ensaio dele entrou em qualquer etapa. Isso
significa tambem que o resultado aqui **nao foi confirmado em dado independente** - a confirmacao
exigiria gastar o conjunto reservado, o que este teste deliberadamente nao faz.

**Erro medio nao e a unica medida que importa.** A Sprint 3 registrou que o algoritmo de menor erro
nao e necessariamente o que deixa passar menos excessos sobre o limite de seguranca: o Gradient
Boosting errava mais em media e, ainda assim, deixava passar menos casos perigosos - 26 contra 36 na
margem de 8 graus. Uma mescla que reduza o MAE pode piorar a decisao contra o limite. A verificacao
desse ponto exige a tabela de limites e fica fora do escopo deste notebook, que mede erro, e nao
seguranca.

### Resultado obtido na execucao de referencia

Os numeros abaixo sao os de uma execucao completa deste notebook, feita em 18 de setembro de 2026,
sobre a base de referencia `_13cat_snull`, e servem para conferir se uma nova execucao reproduz o
mesmo quadro. Eles nao substituem as tabelas acima, que sao recalculadas a cada rodada.

- A verificacao de sanidade **passou** nas sete areas: o Random Forest puro devolveu 2,52 / 3,10 /
  3,89 / 4,22 / 5,72 / 7,53 / 14,57 graus, contra 2,42 / 3,16 / 4,04 / 4,68 / 5,68 / 7,30 / 11,71 da
  entrega da Sprint 3. A unica diferenca grande esta no coil_temps, e ela tem causa conhecida: sao
  as linhas imputadas pelo `_snull`, como a Secao 10 detalha.
- A mescla venceu em **duas** das sete areas: Coil Temps, com pesos `(0; 0,5; 0,5)`, MAE de 14,420
  contra 14,573 do Random Forest puro - ganho de 0,153 grau, ou 1,1%; e IPC Cooling, com os mesmos
  pesos, 3,813 contra 3,888 - ganho de 0,075 grau. Nenhum dos dois chega ao limiar de relevancia que
  o notebook do Extra Trees adota, de 0,10 grau, exceto o do coil_temps, por pouco.
- Nas outras cinco areas o **Random Forest puro foi o melhor resultado**, e nenhuma mescla o superou.
- O **Ridge zerou de novo**: com 9,81 graus de erro agregado contra 6,99 do Random Forest, ele nao
  entra em nenhuma combinacao vencedora, em nenhuma das duas bases.
- A hipotese do peso maior para o Gradient Boosting **nao se confirmou**: o MAE sobe, e nao desce,
  conforme o peso dele aumenta, em todas as sete areas; e o Gradient Boosting puro erra mais que o
  Random Forest puro tambem nas sete.

A leitura que esses numeros sustentam e que a mesclagem nao compensa como regra geral nesta base. As
excecoes sao pequenas e concentradas nas areas de maior dispersao, justamente aquelas em que a media
entre modelos tem mais ruido para cancelar - com a ressalva, ja feita acima, de que o peso foi
escolhido sobre o mesmo conjunto que o avaliou.

**A comparacao entre as bases (Secao 10).** As duas leituras nao apontam para o mesmo lado. Por
contagem de areas da empate: o perfil vence em 2 (ipc_comp, oven_hmi), perde em 2 (coil_temps,
main_vent_sys) e empata em 3; mas por diferenca media do melhor modelo puro ele fica 0,030 grau
**atras**, porque onde perde, perde mais (0,374 em media) do que ganha onde ganha (0,225). No
agregado, o perfil erra menos com as arvores - Random Forest 6,891 contra 6,990 e Gradient Boosting
7,369 contra 7,543 - e **mais** com o Ridge, 10,343 contra 9,811. A decisao sobre agrupar as perdas
por perfil e, portanto, **por area**, e nao uma regra unica.